# 2. Konsumenta A, który zapisuje wiadomości do tabeli "transactions" w bazie Postgres, dodatkowo tabela transactions ma zawierać kolumnę "title_embeding" zawierającą embeding z tytułu przelewu.

In [1]:
import os, sys
import psycopg2
import json
from kafka import KafkaConsumer

sys.path.append(os.path.abspath(os.path.join(os.path.dirname("__file__"), "..")))
from shared.embedding import embed


In [2]:


consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='reda_kafka_lab:9092',
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    group_id='postgres-group',
    value_deserializer=lambda m: json.loads(m.decode('utf-8'))
)


In [19]:
# Połączenie z PostgreSQL
conn = psycopg2.connect(
    dbname='vectordb',
    user='user',
    password='password',
    host='postgres',
    port=5432
)
cur = conn.cursor()



# Pkt 3 Kod SQL definicji tabeli "transactions" zawierający kolumne title_embeding

In [4]:
DDL = """
DROP TABLE IF EXISTS transactions;

CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE transactions (
    id SERIAL PRIMARY KEY,
    sender TEXT NOT NULL,
    receiver TEXT NOT NULL,
    amount NUMERIC(10,2),
    timestamp TIMESTAMP,
    device_sender TEXT NOT NULL,
    device_receiver TEXT NOT NULL,
    title TEXT NOT NULL,
    title_embeding VECTOR(384),
    created_at TIMESTAMP DEFAULT NOW()    
);

CREATE INDEX idx_transactions
ON transactions
USING ivfflat (title_embeding vector_cosine_ops)
WITH (lists = 100);

"""


In [5]:

def initialize_database():
    print("Connecting to database...")

    print("Initializing schema...")

    cur.execute(DDL)
    conn.commit()

    print("Database initialized successfully.")

    cur.execute("SELECT current_database();")
    print(cur.fetchone())

In [6]:
initialize_database()

Connecting to database...
Initializing schema...
Database initialized successfully.
('vectordb',)


# 4. Kod SQL wykonujący operacje INSERT do tabeli transactions.

In [7]:
print(f"Waiting for transfer...")

for msg in consumer:
    data = msg.value
    vectors = embed(data['title'])
    cur.execute(
        "INSERT INTO transactions (sender, receiver, amount, timestamp, device_sender, device_receiver, title, title_embeding) "+
        "VALUES (%s, %s, %s, %s, %s, %s, %s, %s)",
        (data['sender'], data['receiver'], data['amount'], data['timestamp'], data['device_sender'], data['device_receiver'], data['title'], vectors.tolist())
    )
    conn.commit()
    print(f"Inserted into DB: {data}")

Waiting for transfer...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Inserted into DB: {'sender': 'user1', 'receiver': 'user2', 'amount': 120.5, 'timestamp': '2026-05-10T10:00:00', 'device_sender': 'devA', 'device_receiver': 'devB', 'title': 'zwrot za obiad'}
Inserted into DB: {'sender': 'user2', 'receiver': 'user1', 'amount': 50.0, 'timestamp': '2026-05-07T14:00:00', 'device_sender': 'devB', 'device_receiver': 'devA', 'title': 'zwrot za paliwo'}
Inserted into DB: {'sender': 'user3', 'receiver': 'user4', 'amount': 75.0, 'timestamp': '2026-05-09T11:00:00', 'device_sender': 'devC', 'device_receiver': 'devD', 'title': 'oddaje pieniadze'}
Inserted into DB: {'sender': 'user5', 'receiver': 'user6', 'amount': 200.0, 'timestamp': '2026-05-08T09:30:00', 'device_sender': 'devE', 'device_receiver': 'devF', 'title': 'zwrot kosztow'}
Inserted into DB: {'sender': 'user7', 'receiver': 'user8', 'amount': 300.0, 'timestamp': '2026-05-06T16:00:00', 'device_sender': 'devG', 'device_receiver': 'devH', 'title': 'oddaje dlug'}
Inserted into DB: {'sender': 'user9', 'receiver'

KeyboardInterrupt: 

# 6. Zapytania sprawdzające:
a) (SQL) Znajdź przelewy zawierające kontekst:
- jedzenia
- zwrotu środków w zakresie 30 - 150

In [ ]:
#(SQL) Znajdź przelewy zawierające kontekst:
# - jedzenia
# wybralem top 10

vec_food = embed(["jedzenie"])[0].tolist()

vec_food_str = "[" + ",".join(map(str, vec_food)) + "]"

cur.execute(
    """
    SELECT DISTINCT
        sender,
        receiver,
        amount,
        title,
        title_embeding <=> %s::vector AS dist_food        
    FROM transactions
    ORDER BY title_embeding <=> %s::vector
    LIMIT 10;
    """,    
    (vec_food_str, vec_food_str, )
)
results = cur.fetchall()
print("results are...")
for r in results:
    print(">>>", r)

results are...
>>> ('user18', 'user20', Decimal('215.00'), 'jedzenie', 0.0)
>>> ('user14', 'user16', Decimal('175.00'), 'jedzenie', 0.0)
>>> ('user9', 'user11', Decimal('125.00'), 'jedzenie', 0.0)
>>> ('user5', 'user7', Decimal('85.00'), 'czynsz', 0.4906269311904907)
>>> ('user3', 'user5', Decimal('65.00'), 'prezent', 0.4985954165458679)
>>> ('user5', 'user6', Decimal('200.00'), 'zwrot kosztow', 0.5433214255188163)
>>> ('user2', 'user1', Decimal('50.00'), 'zwrot za paliwo', 0.5575752258300781)
>>> ('user3', 'user4', Decimal('75.00'), 'oddaje pieniadze', 0.5626913607120514)
>>> ('user15', 'user17', Decimal('185.00'), 'zakupy', 0.6035970685722184)
>>> ('user9', 'user10', Decimal('20.00'), 'zakupy', 0.6035970685722184)


In [ ]:
#(SQL) Znajdź przelewy zawierające kontekst:
#- zwrotu środków w zakresie 30 - 150
# wybralem top 10

vec_refund = embed(["zwrot środków oddanie pieniędzy zwrot kosztów"])[0].tolist()
vec_refund_str = "[" + ",".join(map(str, vec_refund)) + "]"

cur.execute(
    """
    SELECT DISTINCT
        sender,
        receiver,
        amount,
        title,
        title_embeding <=> %s::vector AS dist_refund        
    FROM transactions
    WHERE amount BETWEEN 30 AND 150    
    ORDER BY title_embeding <=> %s::vector
    
    LIMIT 10;
    """,    
    (vec_refund_str, vec_refund_str,)
)
results = cur.fetchall()
print("results are...")
for r in results:
    print(">>>", r)

results are...
>>> ('user1', 'user2', Decimal('120.50'), 'zwrot za obiad', 0.381250835582563)
>>> ('user2', 'user1', Decimal('50.00'), 'zwrot za paliwo', 0.420681357383728)
>>> ('user3', 'user4', Decimal('75.00'), 'oddaje pieniadze', 0.44642043113708496)
>>> ('user5', 'user7', Decimal('85.00'), 'czynsz', 0.47913163900375366)
>>> ('user9', 'user11', Decimal('125.00'), 'jedzenie', 0.5411010980606079)
>>> ('user10', 'user12', Decimal('135.00'), 'zakupy', 0.6055345770398027)
>>> ('user15', 'user16', Decimal('80.00'), 'zakupy', 0.6055345770398027)
>>> ('user6', 'user8', Decimal('95.00'), 'zakupy', 0.6055345770398027)
>>> ('user3', 'user5', Decimal('65.00'), 'prezent', 0.6861692667007446)
>>> ('user1', 'user3', Decimal('45.00'), 'pizza', 0.7223427891731262)


In [22]:
#(SQL) Znajdź przelewy zawierające kontekst:
# - jedzenia
# AND
#- zwrotu środków w zakresie 30 - 150



vec_refund = embed(["zwrot środków oddanie pieniędzy zwrot kosztów"])[0].tolist()
vec_food = embed(["jedzenie obiad pizza zakupy spożywcze"])[0].tolist()

vec_refund_str = "[" + ",".join(map(str, vec_refund)) + "]"
vec_food_str = "[" + ",".join(map(str, vec_food)) + "]"

cur.execute(
    """
    SELECT DISTINCT
        sender,
        receiver,
        amount,
        title,
        title_embeding <=> %s::vector AS dist_refund,
        title_embeding <=> %s::vector AS dist_food,
        (title_embeding <=> %s::vector) + (title_embeding <=> %s::vector) AS total_distance
    FROM transactions
    WHERE amount BETWEEN 30 AND 150
    AND title_embeding <=> %s::vector < 0.75
    AND title_embeding <=> %s::vector < 0.75
    ORDER BY total_distance
    
    LIMIT 10;
    """,    
    (vec_refund_str, vec_food_str, vec_refund_str, vec_food_str, vec_refund_str, vec_food_str,)
)
results = cur.fetchall()
print("results are...")
for r in results:
    print(">>>", r)

results are...
>>> ('user1', 'user2', Decimal('120.50'), 'zwrot za obiad', 0.381250835582563, 0.39688488226441865, 0.7781357178469817)
>>> ('user9', 'user11', Decimal('125.00'), 'jedzenie', 0.5411010980606079, 0.3051005965591139, 0.8462016946197218)
>>> ('user2', 'user1', Decimal('50.00'), 'zwrot za paliwo', 0.420681357383728, 0.4514889659272353, 0.8721703233109633)
>>> ('user3', 'user4', Decimal('75.00'), 'oddaje pieniadze', 0.44642043113708496, 0.46374696060978515, 0.9101673917468701)
>>> ('user10', 'user12', Decimal('135.00'), 'zakupy', 0.6055345770398027, 0.4511067540113095, 1.0566413310511122)
>>> ('user15', 'user16', Decimal('80.00'), 'zakupy', 0.6055345770398027, 0.4511067540113095, 1.0566413310511122)
>>> ('user6', 'user8', Decimal('95.00'), 'zakupy', 0.6055345770398027, 0.4511067540113095, 1.0566413310511122)
>>> ('user5', 'user7', Decimal('85.00'), 'czynsz', 0.47913163900375366, 0.5852764350505364, 1.06440807405429)
>>> ('user1', 'user3', Decimal('45.00'), 'pizza', 0.72234278